# 46장 · 검색 규칙 변경과 실패 단계 진단

검색 규칙을 바꾼 뒤에는 문서가 다시 검색되는지와 그 문서로 옳은 답을 만드는지를 따로 검사합니다.

기준선에서 실패한 Q03을 대상으로 ‘실패’와 ‘오류’를 검색 규칙에 추가합니다. 입력 문서, 생성 함수, k=2를 고정하고 p3 검색 여부와 인용 결과를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요.


## 기준선에서 놓친 질문만 다시 검색하기

45장에서 정답이 있는데도 검색하지 못한 행을 가져옵니다. 실패·오류 문의에만 p3을 찾는 실험 후보를 추가해 원래 검색 결과와 나란히 남깁니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/midterm/chapter45.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter46-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/midterm/chapter45.json"
sample_file = here / "chapter46-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['runs'])

교재 예시 chapter46-start.json
[{'id': 'Q01', 'question': '과제는 어디에 내나요?', 'action': '답변', 'expected': ['p2'], 'hits': ['p2'], 'passed': True}, {'id': 'Q02', 'question': '내 점수는 몇 점인가요?', 'action': '보류', 'expected': [], 'hits': [], 'passed': True}, {'id': 'Q03', 'question': '과제 업로드가 실패했어요. 어떻게 해야 하나요?', 'action': '답변', 'expected': ['p3'], 'hits': [], 'passed': False}]


두 번째 줄은 읽은 파일의 `runs` 값입니다.

## 키워드 변경 전후의 문단 적중

완성 실험 기록｜변경 전 Q03은 검색 1회, 빈 목록, 생성 0회입니다. 후보에서는 같은 문장에 오류 낱말을 인식해 p3을 가져옵니다. 여기서는 제공 기록에 맞춰 검색 조건 한 가지를 재현합니다.


### 셀 01 · 키워드 규칙 변경과 적중

Q03에 어떤 낱말이 들어 있으며 그 조건에서 어느 문단을 돌려주는지 확인하세요.


In [1]:
error_doc = {'id': 'course-guide-v1:p3',
 'text': '제출 오류가 발생하면 오류 화면과 발생 시간을 기록하고 담당 강사에게 문의합니다. 비밀키가 보이는 화면은 공유하지 '
         '않습니다.',
 'source': '가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html)',
 'page': 1}
question = '과제 업로드가 실패했어요. 어떻게 해야 하나요?'
found = [error_doc] if any(word in question for word in ('오류','실패')) else []
print([d['id'] for d in found])

['course-guide-v1:p3']


course-guide-v1:p3을 찾았습니다. 원문에는 오류 화면과 발생 시간 기록, 강사 문의, 비밀키 공유 금지가 담겨 있습니다.


## 검색 성공 뒤의 인용 실패

고정 생성기는 Q01에서 썼던 p2를 인용합니다. 검색은 p3을 돌려주는데 p2 인용을 사용자에게 보여 주면 출처가 어긋납니다. 그래서 이번에는 기준선을 유지하고 생성 수정을 별도의 실험으로 넘깁니다.


### 셀 02 · 검색 이후 실패 단계 비교

두 행은 같은 Q03의 제공 로컬 실행 기록입니다. 검색 정책 외에 무엇을 고정했는지 기억하며 살펴보세요.

Q03에서 검색이 허용한 문단은 p3인데 고정 답변의 대괄호 안에는 p2가 있습니다. ‘검색 성공’에는 p3을, ‘인용 검사 실패’에는 p2를 적으세요. 검색 문서를 찾았다는 이유로 이 답을 사용자에게 내보내지 않습니다.


In [2]:
before = {'retrieved_ids': [],
 'result': {'status': 'no_evidence',
            'answer': '제공된 자료에서 확인할 수 없습니다.',
            'sources': [],
            'generation_calls': 0,
            'retrieved_ids': [],
            'seconds': 1e-06}}
after = {'retrieved_ids': ['course-guide-v1:p3'],
 'result': {'status': 'invalid_citation',
            'answer': '출처 표시를 확인해야 합니다.',
            'sources': [],
            'generation_calls': 1,
            'retrieved_ids': ['course-guide-v1:p3'],
            'seconds': 4e-06}}
print(before['retrieved_ids'], before['result']['status'])
print(after['retrieved_ids'], after['result']['status'])

[] no_evidence
['course-guide-v1:p3'] invalid_citation


빈 목록의 no_evidence가 p3과 invalid_citation으로 바뀌었습니다. 검색 성공과 사용자가 읽을 수 있는 답변 성공은 따로 기록해야 합니다. 자기 팀도 가장 자주 틀리는 단계 하나부터 비교하세요.

수정 전의 no_evidence와 수정 뒤의 invalid_citation은 서로 다른 결함입니다. 두 번째 실행은 p3을 찾는 데 성공했습니다. 이 상태에서 검색어 규칙을 다시 넓히기보다 어떤 인용 ID를 답변에 넣었는지 확인하면 다음 수정 범위를 좁힐 수 있습니다.


#### 검색을 고쳤는데 답변은 아직 실패한다면

<mark>기대 문단을 찾아낸 시점과 올바른 문단을 인용한 시점을 구분합니다.</mark> 같은 질문의 상태가 no_evidence에서 invalid_citation으로 바뀌었다면 다음에 읽을 곳은 답변의 출처 필드입니다. 팀 기록에도 두 단계의 결과를 따로 남기세요.

선수 개념 · 45장 질문 전체 일치 검색의 기준선과 빈 결과

### 다음 실패가 드러나는 시점

변경 전에는 근거가 없어 생성을 0회로 마쳤습니다. 변경 후에는 p3을 찾아 생성까지 1회 실행했지만 인용 검사에서 멈췄습니다. 두 상태를 나란히 읽으면 다음에 고칠 대상이 검색 키워드에서 인용으로 바뀐 것을 알 수 있습니다.


## Q03을 고친 뒤 처음 보는 표현으로 다시 시험하기

후보 검색기가 Q03에서 p3을 찾았습니다. 팀이 이미 이 질문을 보며 “실패”를 낱말 규칙에 추가했다면, Q03 하나로 새로운 문의에도 통한다고 주장하기 어렵습니다.

### 셀 03 · 같은 수정에 다른 표현 하나 넣기

기존 Q03과 별도로 동료가 준비한 문장을 입력합니다. 물어본 뜻과 검색 규칙에 적힌 낱말을 구분해서 보세요.

In [3]:
def find_upload_help(question_text):
    if any(word in question_text for word in ("오류", "실패")):
        return [error_doc["id"]]
    return []

print(find_upload_help(question))
unseen = "과제를 올리는데 화면이 멈춰요."
print(find_upload_help(unseen))

['course-guide-v1:p3']
[]


첫 질문은 p3을 찾지만 “화면이 멈춰요”에는 두 낱말이 없어 빈 결과가 나옵니다. 새 문장을 고치는 데 사용한 순간부터 그 문장도 다음 판의 개발 자료로 옮겨야 합니다.

### 팀 실험 노트에 남길 판단

<table><thead><tr><th scope="col">입력</th><th scope="col">관찰값</th><th scope="col">이번 결정</th></tr></thead><tbody><tr><td>공개한 Q03</td><td>후보 p3, 기존 생성기는 p2 인용</td><td>검색 수정과 인용 수정을 나눠 추적</td></tr><tr><td>새 문의 “화면이 멈춰요”</td><td>후보 0개</td><td>수정 전 첫 관찰로 보존하고 추가 실험 설계</td></tr></tbody></table>

팀의 후보가 새 문장에도 성공했다면 어떤 문서가 몇 번째였는지 적으세요. 이후 규칙을 더 고치면 같은 질문은 회귀 검사에 사용하고, 처음 보는 표현은 별도로 확보합니다.

<mark>이미 수정에 사용한 질문이 통과한 결과는 회귀 근거로 보관합니다.</mark> Q03의 성공은 그 질문이 다시 깨지는지 확인하는 데 쓸 수 있습니다. 새로운 표현에 대한 성능은 그 표현을 넣어 본 결과로 판단합니다.

47장의 실행 흐름을 바꿀 때에도 검색 후보와 인용을 각각 기록하세요.

## 팀 사례에 적용하기 · 기준선에서 놓친 질문만 다시 검색하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
candidate_rule = {"terms": ["오류", "실패"], "document": "p3"}
def try_rule(text, previous_ids):
    target = candidate_rule["document"]
    if target in incoming["documents"] and any(word in text for word in candidate_rule["terms"]):
        return [target]
    return previous_ids

trials = [{**row, "candidate": try_rule(row["question"], row["hits"])} for row in incoming["runs"]]
new = {"runs": trials, "documents": incoming["documents"], "decision": "기준선 유지",
       "candidate_rule": candidate_rule, "unseen_hits": try_rule("과제를 올리는데 화면이 멈춰요.", [])}
print([(row["id"], row["hits"], row["candidate"]) for row in trials])

[('Q01', ['p2'], ['p2']), ('Q02', [], []), ('Q03', [], ['p3'])]


같은 Q03이 개선돼도 ‘화면이 멈춰요’는 놓칩니다. 기존 기준선은 그대로 유지하고, 후보 규칙과 실험 결과를 따로 넘겨 이후 판단에 사용합니다.

In [ ]:
output_file = project_root / "team-project/midterm/chapter46.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/midterm/chapter46.json


47장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

이 실험의 선택 후보는 error_keyword, 실제 유지한 방식은 baseline입니다. 다음 장을 맡은 팀원이 같은 조건에서 시작할 수 있도록 유지한 방식과 그 이유를 함께 건네세요.
